# Regresión logística — Supuestos y validación
Breast Cancer Wisconsin (sklearn). Objetivo: 1 = maligno.

A1 independencia · A2 linealidad del logit · A3 multicolinealidad · A4 observaciones influyentes · A5 tamaño de muestra/separación · bondad de ajuste

In [ ]:
# !pip install statsmodels scikit-learn

In [ ]:
import numpy as np, pandas as pd, statsmodels.api as sm
from scipy import stats
from sklearn.datasets import load_breast_cancer
from statsmodels.stats.outliers_influence import variance_inflation_factor

data = load_breast_cancer(as_frame=True)
X, y = data.data, (data.target == 0).astype(int)      # 1 = maligno

# A3 · Multicolinealidad: VIF > 10 → problema serio
cand = ['mean radius', 'mean perimeter', 'mean area', 'mean texture',
        'mean smoothness', 'mean symmetry']
Xc = sm.add_constant(X[cand])
vif = {c: variance_inflation_factor(Xc.values, i)
       for i, c in enumerate(Xc.columns) if c != 'const'}
print(pd.Series(vif).round(1))       # radio 413, perímetro 412, área 41

# A2 · Linealidad del logit: Box–Tidwell (H0: coef. de x·ln x = 0)
keep = ['mean radius', 'mean texture', 'mean smoothness', 'mean symmetry']
def box_tidwell(df):
    Z = df.copy()
    for c in df.columns: Z[c + '_xlnx'] = df[c] * np.log(df[c])
    fit = sm.Logit(y, sm.add_constant(Z)).fit(disp=0)
    return fit.pvalues.filter(like='_xlnx')
print(box_tidwell(X[keep]).round(3))         # textura p < 0.001 → transformar
X2 = X[keep].copy()
X2['mean texture'] = 1 / X2['mean texture']   # 1/x linealiza el logit
X2 = X2.rename(columns={'mean texture': 'inv texture'})
print(box_tidwell(X2).round(3))               # todos p > 0.05 ✓


In [ ]:
from statsmodels.stats.stattools import durbin_watson
model = sm.GLM(y, sm.add_constant(X2), family=sm.families.Binomial()).fit()
print(model.summary())                  # pruebas z de Wald e IC 95 %

# Odds ratios por 1 DE (en unidades crudas son absurdos, p. ej. 3.7e+56)
Z = sm.add_constant((X2 - X2.mean()) / X2.std())
zfit = sm.GLM(y, Z, family=sm.families.Binomial()).fit()
print(np.exp(zfit.params[1:]).round(2))

# Prueba global: razón de verosimilitud vs. modelo nulo
lr = model.null_deviance - model.deviance
print('LR =', round(lr, 1), 'p =', stats.chi2.sf(lr, model.df_model))

# A4 · Influencia: D de Cook, apalancamiento, residuos estudentizados
infl = model.get_influence()
cook = infl.cooks_distance[0]
print("Cook's D > 4/n:", (cook > 4 / len(y)).sum())

# A1 · Independencia (si las filas tienen orden/tiempo): DW ≈ 2
print('DW =', round(durbin_watson(model.resid_deviance), 2))

# Bondad de ajuste: Hosmer–Lemeshow (H0: el modelo ajusta bien)
def hosmer_lemeshow(y, p, g=10):
    d = pd.DataFrame({'y': y, 'p': p})
    d['g'] = pd.qcut(d.p.rank(method='first'), g, labels=False)
    t = d.groupby('g').agg(o=('y', 'sum'), e=('p', 'sum'), n=('y', 'count'))
    hl = ((t.o - t.e) ** 2 / (t.e * (1 - t.e / t.n))).sum()
    return round(hl, 2), round(stats.chi2.sf(hl, g - 2), 3)
print(hosmer_lemeshow(y, model.fittedvalues))


## Diagnóstico gráfico

In [ ]:
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
# A1: residuos vs. orden
ax[0].scatter(range(len(y)), model.resid_deviance, s=8); ax[0].axhline(0, c='k'); ax[0].set_title('Residuos de devianza vs. orden')
# A4: distancia de Cook
ax[1].stem(cook, markerfmt=','); ax[1].axhline(4/len(y), c='r', ls='--'); ax[1].set_title('Distancia de Cook')
# Ajuste: curva de calibración
fy, fx = calibration_curve(y, model.fittedvalues, n_bins=10, strategy='quantile')
ax[2].plot([0, 1], [0, 1], '--', c='gray'); ax[2].plot(fx, fy, 'o-'); ax[2].set_title('Curva de calibración')
plt.tight_layout(); plt.show()

## Gráficos de logit empírico (A2)

In [ ]:
def empirical_logit(x, y, bins=10, ax=None):
    q = pd.qcut(x, bins, duplicates='drop')
    g = pd.DataFrame({'x': x, 'y': y}).groupby(q, observed=True)
    k, m = g.y.sum(), g.y.count()
    ax.plot(g.x.mean(), np.log((k + .5) / (m - k + .5)), 'o-')
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for a, (name, v) in zip(ax, [('radio medio', X['mean radius']), ('textura media', X['mean texture']), ('1 / textura', 1 / X['mean texture'])]):
    empirical_logit(v, y, ax=a); a.set_title(name); a.set_ylabel('log-odds empírico')
plt.tight_layout(); plt.show()

## Evaluación en conjunto de prueba

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
Xtr, Xte, ytr, yte = train_test_split(sm.add_constant(X2), y, test_size=.3, stratify=y, random_state=42)
m = sm.Logit(ytr, Xtr).fit(disp=0); p = m.predict(Xte)
print('AUC =', round(roc_auc_score(yte, p), 3))
print(confusion_matrix(yte, p >= .5)); print(classification_report(yte, p >= .5))